# ForgeLM — GRPO Reasoning RL

Train a model to reason step-by-step using Group Relative Policy Optimization (the method behind DeepSeek-R1).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/grpo_reasoning.ipynb)

In [ ]:
# Install ForgeLM (clean install to avoid Colab dependency conflicts)
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git
!pip uninstall -y wandb -q  # Remove conflicting wandb (not needed, tensorboard is default)

In [ ]:
# GRPO dataset: prompt only (model generates responses during training)
import json

prompts = [
    {"prompt": "What is 15% of 240? Think step by step."},
    {"prompt": "If a train travels 60 km/h for 2.5 hours, how far does it go?"},
    {"prompt": "What is the sum of the first 10 prime numbers?"},
    {"prompt": "A rectangle has width 5 and perimeter 22. What is the length?"},
]

with open("math_prompts.jsonl", "w") as f:
    for p in prompts:
        f.write(json.dumps(p) + "\n")

print(f"Created {len(prompts)} GRPO prompts")

In [ ]:
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 512
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "grpo"              # Reasoning RL
  grpo_num_generations: 4           # Generate 4 responses per prompt
  grpo_max_new_tokens: 256
  output_dir: "./grpo_checkpoints"
  num_train_epochs: 1
  per_device_train_batch_size: 1
  learning_rate: 1.0e-6             # Very low LR for RL stability

data:
  dataset_name_or_path: "math_prompts.jsonl"
"""

with open("grpo_config.yaml", "w") as f:
    f.write(config_yaml)
print("Config saved!")

In [ ]:
!forgelm --config grpo_config.yaml --dry-run

In [ ]:
!forgelm --config grpo_config.yaml

## How GRPO Works

1. For each prompt, the model generates `num_generations` responses
2. Responses are scored by a reward function (correctness for math)
3. The model is updated to prefer higher-scoring responses
4. No human preference data needed — the model learns from its own outputs

This is the method behind DeepSeek-R1's reasoning capabilities.